# A03 · Optimization: SGD, Momentum & Adam

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adib2024/aws-architect-path/blob/main/notebooks/a03_optimizers.ipynb)

Training is iterative numerical optimisation. Learn exactly what each optimiser does to every parameter, why the learning rate has a hard stability limit, and why every LLM trains with AdamW plus warmup and cosine decay.

Part of the **AI Engineer Path** (https://adib2024.github.io/aws-architect-path/#a03). In Colab, use **Runtime → Change runtime type → T4 GPU** for anything that trains.

**You will:**
- Derive the stability limit $\eta < 2/\lambda_{\max}$ for gradient descent on a quadratic.
- Write SGD, momentum and Adam updates by hand and match `torch.optim`.
- Explain bias correction in Adam and decoupled weight decay in AdamW.
- Build a warmup + cosine learning-rate schedule.

## The math

### Stability limit of gradient descent

$$f(x) = \tfrac{1}{2}\lambda x^2 \;\Rightarrow\; x_{t+1} = (1 - \eta\lambda)\,x_t$$

The iterate shrinks only if $|1 - \eta\lambda| < 1$, i.e. $0 < \eta < 2/\lambda$. In many dimensions the largest curvature $\lambda_{\max}$ sets the limit.

**Worked example**

1. $f(x,y) = \tfrac12(x^2 + 25y^2)\Rightarrow \lambda = 1, 25$ (condition number 25)
2. $\eta_{\max} = 2/25 = 0.08$ (η = 0.081 diverges along y)
3. at $\eta = 0.07$: the x error shrinks only by $1 - 0.07 = 0.93$ per step (slow along the flat direction: the core problem momentum and Adam solve)

### Momentum (heavy ball)

$$v_{t+1} = \mu v_t + \nabla L(\theta_t), \qquad \theta_{t+1} = \theta_t - \eta\, v_{t+1}$$

With constant gradient $g$, velocity converges to $g/(1-\mu)$: $\mu = 0.9$ gives a 10× larger effective step along consistent directions.

**Worked example**

1. $f(x)=x^2,\ x_0=3,\ \eta=0.1,\ \mu=0.9$
2. $g_0 = 6,\ v_1 = 6,\ x_1 = 3 - 0.6 = 2.4$
3. $g_1 = 4.8,\ v_2 = 0.9\cdot 6 + 4.8 = 10.2,\ x_2 = 2.4 - 1.02 = 1.38$ (plain SGD would be at 1.92)

### Adam with bias correction

$$\begin{aligned} m_t &= \beta_1 m_{t-1} + (1-\beta_1)\,g_t \\ v_t &= \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2 \\ \hat m_t &= \frac{m_t}{1-\beta_1^t},\quad \hat v_t = \frac{v_t}{1-\beta_2^t} \\ \theta_t &= \theta_{t-1} - \eta\,\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon} \end{aligned}$$

First step with $\theta=1,\ g=0.5,\ \eta=0.1,\ \beta_1=0.9,\ \beta_2=0.999$.

**Worked example**

1. $m_1 = 0.1\cdot0.5 = 0.05,\quad v_1 = 0.001\cdot0.25 = 0.00025$ (biased towards 0)
2. $\hat m_1 = 0.05/0.1 = 0.5,\quad \hat v_1 = 0.00025/0.001 = 0.25$ (bias correction undoes the zero init)
3. $\theta_1 = 1 - 0.1\cdot 0.5/\sqrt{0.25} = 0.9$ (the first step has size $\eta$ whatever the gradient's scale)

### Warmup + cosine schedule

$$\eta_t = \begin{cases} \eta_{\max}\,\dfrac{t}{T_w} & t < T_w \\[4pt] \eta_{\min} + \tfrac12(\eta_{\max}-\eta_{\min})\left(1+\cos\dfrac{\pi(t-T_w)}{T-T_w}\right) & t \ge T_w \end{cases}$$

Typical LLM values: $\eta_{\max} = 3\times10^{-4}$ (smaller for bigger models), warmup 1–2% of steps, $\eta_{\min} = 0.1\,\eta_{\max}$.

**Worked example**

1. $T_w = 100,\ T = 1000$: at $t=50$, $\eta = 0.5\,\eta_{\max}$
2. at $t = 550$ (halfway through decay): $\eta = \eta_{\min} + 0.5(\eta_{\max}-\eta_{\min}) = 0.55\,\eta_{\max}$

## Lab: Watch optimisers fail and succeed

Find the divergence threshold by experiment, implement Adam by hand, and build the LLM learning-rate schedule.

### Step 1

Race three optimisers on an ill-conditioned quadratic. Then set SGD's lr to 0.081 and watch it blow up.

In [ ]:
import torch, math
def f(p):
    x, y = p
    return 0.5 * (x**2 + 25 * y**2)          # condition number 25 -> SGD needs lr < 0.08
def run(name, lr, steps=100):
    p = torch.tensor([-4.0, 1.5], requires_grad=True)
    opt = {'sgd': lambda: torch.optim.SGD([p], lr=lr),
           'momentum': lambda: torch.optim.SGD([p], lr=lr, momentum=0.9),
           'adam': lambda: torch.optim.Adam([p], lr=lr)}[name]()
    for _ in range(steps):
        opt.zero_grad(); loss = f(p); loss.backward(); opt.step()
    return loss.item()
for name, lr in [('sgd', 0.07), ('sgd', 0.081), ('momentum', 0.02), ('adam', 0.1)]:
    print(f"{name:9s} lr={lr:<6} final loss={run(name, lr):.3e}")

### Step 2

Implement Adam yourself and check it matches `torch.optim.Adam` to floating-point precision.

In [ ]:
def adam_step(theta, g, m, v, t, lr=0.1, b1=0.9, b2=0.999, eps=1e-8):
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g * g
    m_hat = m / (1 - b1 ** t)
    v_hat = v / (1 - b2 ** t)
    return theta - lr * m_hat / (v_hat.sqrt() + eps), m, v

theta = torch.tensor([-4.0, 1.5]); m = torch.zeros(2); v = torch.zeros(2)
p = theta.clone().requires_grad_(); opt = torch.optim.Adam([p], lr=0.1)
for t in range(1, 51):
    g = torch.tensor([theta[0], 25 * theta[1]])          # analytic gradient of f
    theta, m, v = adam_step(theta, g, m, v, t)
    opt.zero_grad(); f(p).backward(); opt.step()
print(theta, p.data, torch.allclose(theta, p.data, atol=1e-6))

### Step 3

Build warmup + cosine with `LambdaLR` and print a few values.

In [ ]:
model = torch.nn.Linear(10, 1)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)
warmup, total, min_ratio = 100, 1000, 0.1
def lr_lambda(step):
    if step < warmup:
        return (step + 1) / warmup
    progress = (step - warmup) / (total - warmup)
    return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * progress))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
lrs = []
for step in range(total):
    opt.step(); sched.step(); lrs.append(sched.get_last_lr()[0])
print([f"{lrs[i]:.2e}" for i in (0, 49, 99, 550, 999)])

### Clean up

Nothing to clean up.

## Check yourself

**1. Gradient descent on $f = \tfrac12(x^2 + 100y^2)$. Largest stable learning rate?**

A. 2  
B. 0.2  
C. 0.02  
D. 0.01

<details><summary>Answer</summary>

**C.** $\eta < 2/\lambda_{\max} = 2/100 = 0.02$.

</details>

**2. At step 1, Adam's $m_1 = 0.1 g$ without bias correction. What does bias correction do?**

A. Nothing at step 1  
B. Divides by $1-\beta_1^1 = 0.1$, restoring $\hat m_1 = g$  
C. Adds weight decay  
D. Clips the gradient

<details><summary>Answer</summary>

**B.** Moments start at zero and are biased low early on. Dividing by $1-\beta^t$ corrects that, and the correction fades as $t$ grows.

</details>

**3. Why is AdamW preferred over Adam with L2 regularisation for transformers?**

A. It uses less memory  
B. Decay is decoupled from the adaptive scaling, so every weight decays at the same relative rate  
C. It doesn't need a learning rate  
D. It's faster per step

<details><summary>Answer</summary>

**B.** With L2 in the gradient, the decay term gets divided by $\sqrt{v}$, so weights with large gradients are barely regularised. AdamW applies decay directly.

</details>